In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def crawl_ecommerce_products(url):
    # Set headers to mimic a browser request and avoid getting blocked
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            print(f"Failed to retrieve the webpage. Status code: {response.status_code}")
            return None

        soup = BeautifulSoup(response.content, 'html.parser')

        # List to store extracted product details
        products = []

        # Note: These CSS selectors are generic templates.
        # To crawl a specific e-commerce site, you must inspect its HTML and update these selectors.
        product_cards = soup.select('.product-card, .product-item, .thumbnail')

        if not product_cards:
            # Fallback mock search for demo purposes
            print("No product cards found with the primary selector. Attempting alternative elements...")
            product_cards = soup.find_all('div', class_=lambda x: x and ('product' in x or 'item' in x))

        for card in product_cards:
            # Extract Title
            title_el = card.find(['h2', 'h3', 'a'], class_=lambda x: x and 'title' in x) or card.find('a')
            title = title_el.text.strip() if title_el else 'N/A'

            # Extract Link
            link = ''
            if title_el and title_el.name == 'a' and title_el.has_attr('href'):
                link = title_el['href']
            else:
                link_el = card.find('a', href=True)
                link = link_el['href'] if link_el else 'N/A'

            # Resolve relative links
            if link.startswith('/'):
                from urllib.parse import urljoin
                link = urljoin(url, link)

            # Extract Price
            price_el = card.find(class_=lambda x: x and 'price' in x)
            price = price_el.text.strip() if price_el else 'N/A'

            if title != 'N/A' or price != 'N/A':
                products.append({
                    'Product Name': title,
                    'Price': price,
                    'Product Link': link
                })

        return pd.DataFrame(products)

    except Exception as e:
        print(f"An error occurred during crawling: {e}")
        return None

In [2]:
# Real-life example sandbox website designed for testing web crawlers
target_url = 'http://books.toscrape.com/'

import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

def crawl_books_sandbox(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            print(f"Failed to load page. Status: {response.status_code}")
            return None

        soup = BeautifulSoup(response.content, 'html.parser')
        products = []

        # Exact CSS selector for products on books.toscrape.com
        book_cards = soup.select('article.product_pod')

        for book in book_cards:
            # Extract Title from the image alt attribute or title attribute of the link
            title_el = book.select_one('h3 a')
            title = title_el['title'] if title_el and title_el.has_attr('title') else (title_el.text.strip() if title_el else 'N/A')

            # Extract Link
            link_raw = title_el['href'] if title_el and title_el.has_attr('href') else ''
            link = urljoin(url, link_raw) if link_raw else 'N/A'

            # Extract Price
            price_el = book.select_one('p.price_color')
            price = price_el.text.strip() if price_el else 'N/A'

            # Extract Availability/Stock status
            availability_el = book.select_one('p.availability')
            availability = availability_el.text.strip() if availability_el else 'N/A'

            products.append({
                'Product Name': title,
                'Price': price,
                'Availability': availability,
                'Product Link': link
            })

        return pd.DataFrame(products)
    except Exception as e:
        print(f"Error crawling page: {e}")
        return None

print(f"Starting crawler for real-world sandbox: {target_url}")
product_df = crawl_books_sandbox(target_url)

if product_df is not None and not product_df.empty:
    display(product_df.head(10))
else:
    print("No product data was extracted. Check connection or selectors.")

Starting crawler for real-world sandbox: http://books.toscrape.com/


,Product Name,Price,Availability,Product Link
0,A Light in the Attic,£51.77,In stock,http://books.toscrape.com/catalogue/a-light-in...
1,Tipping the Velvet,£53.74,In stock,http://books.toscrape.com/catalogue/tipping-th...
2,Soumission,£50.10,In stock,http://books.toscrape.com/catalogue/soumission...
3,Sharp Objects,£47.82,In stock,http://books.toscrape.com/catalogue/sharp-obje...
4,Sapiens: A Brief History of Humankind,£54.23,In stock,http://books.toscrape.com/catalogue/sapiens-a-...
5,The Requiem Red,£22.65,In stock,http://books.toscrape.com/catalogue/the-requie...
6,The Dirty Little Secrets of Getting Your Dream...,£33.34,In stock,http://books.toscrape.com/catalogue/the-dirty-...
7,The Coming Woman: A Novel Based on the Life of...,£17.93,In stock,http://books.toscrape.com/catalogue/the-coming...
8,The Boys in the Boat: Nine Americans and Their...,£22.60,In stock,http://books.toscrape.com/catalogue/the-boys-i...
9,The Black Maria,£52.15,In stock,http://books.toscrape.com/catalogue/the-black-...
